In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


"""STEP 3: rules file + world  ->  the answer key

Runs the rules engine (engine/rules_engine.py) on every boot and every order.
Output World/ground_truth.csv   one row per boot: code, excluded share, RVC, verdict, reason
       World/order_truth.csv    one row per order: verdict on its ship date, duty payable

These are the "right answers" the assistant is scored against later.
Run:  python step3_answer_key.py
"""
import csv, pathlib, sys
from collections import Counter
from datetime import date

# ---- FOLDERS: set the project location once, build everything from it ----
from pathlib import Path
REPO_ROOT        = Path('/content/drive/MyDrive/woodb-agent-project')   # your copy of the repo
CUSMA_ROOT       = REPO_ROOT / 'prototypes' / 'cusma_qualification'
CATALOGUE_FOLDER = CUSMA_ROOT / 'Catalogue'
ENGINE_FOLDER    = CUSMA_ROOT / 'Engine'
RULES_FOLDER     = CUSMA_ROOT / 'Rules'
WORLD_FOLDER     = CUSMA_ROOT / 'World'

# ---- LOAD THE ENGINE ----
sys.path.insert(0, str(ENGINE_FOLDER))   # tell Python to also look in the Engine folder
import rules_engine as eng               # the engine itself reads Rules/footwear_rules.json and World/*.csv


AS_OF = date(2026, 11, 1)




def main():
    boots = []
    for sku in eng.SKUS:
        r = eng.check_origin(sku, AS_OF)["data"]
        boots.append({"sku": sku, "variant_of": eng.SKUS[sku].get("variant_of", ""), "hs_code": r["hs_code"] or "",
                      "excluded_share": r["tariff_shift"]["excluded_share"], "tariff_shift_ok": r["tariff_shift"]["passed"],
                      "rvc": r["rvc"]["value"], "verdict": r["verdict"], "reason": r["escalate_reason"] or ""})
    orders = []
    with open(WORLD_FOLDER / "orders.csv", newline="") as f:
        for o in csv.DictReader(f):
            ship = date.fromisoformat(o["ship_date"])
            d = eng.duty(o["sku"], int(o["quantity"]), float(o["unit_price_usd"]), ship)
            v = eng.check_origin(o["sku"], ship)["data"]["verdict"]
            orders.append({**o, "verdict_on_ship_date": v,
                           "duty_payable_usd": d["data"]["duty_payable_usd"] if d["ok"] else "",
                           "duty_if_not_qualifying_usd": d["data"]["duty_if_not_qualifying_usd"] if d["ok"] else "",
                           "note": "" if d["ok"] else d["error"]["message"]})
    for name, rows in (("ground_truth.csv", boots), ("order_truth.csv", orders)):
        with open(WORLD_FOLDER / name, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)

    print(f"Answer key written: {len(boots)} boots, {len(orders)} orders (rules version {eng.RULES['rules_version']})")
    print("Boot verdicts:", dict(Counter(b["verdict"] for b in boots)))
    changed = sum(1 for o in orders if o["verdict_on_ship_date"] != next(b["verdict"] for b in boots if b["sku"] == o["sku"]))
    print(f"Orders whose verdict changes by ship date (e.g. a declaration expiring): {changed}")


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None, 'display.max_colwidth', None)
W = str(WORLD_FOLDER) + '/'
display(pd.read_csv(W + 'ground_truth.csv'))
display(pd.read_csv(W + 'order_truth.csv'))

#blanks instead of NaNs
#display(pd.read_csv(W + 'ground_truth.csv').fillna(''))
#display(pd.read_csv(W + 'order_truth.csv').fillna(''))